In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/bronislawpodhajski/movies/IMDB Dataset.csv


In [2]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/bronislawpodhajski/movies/IMDB Dataset.csv")
display(df.head())
print(df.shape)

df = df.drop('sentiment', axis=1)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


(50000, 2)


In [3]:
print(df.shape)

(50000, 1)


In [4]:
import re
import unicodedata
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt')
stop_words = stopwords.words('english')

# Converts the unicode file to ascii
def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn')

def preprocess_sentence(w):
    w = unicode_to_ascii(w.lower().strip())
    # creating a space between a word and the punctuation following it
    # eg: "he is a boy." => "he is a boy ."
    w = re.sub(r"([?.!,¿])", r" \1 ", w)
    w = re.sub(r'[" "]+', " ", w)
    # replacing everything with space except (a-z, A-Z, ".", "?", "!", ",")
    w = re.sub(r"[^a-zA-Z?.!]+", " ", w)
    w = re.sub(r'\b\w{0,2}\b', '', w)

    # remove stopword
    mots = word_tokenize(w.strip())
    mots = [mot for mot in mots if mot not in stop_words]
    return ' '.join(mots).strip()

df.review = df.review.apply(lambda x :preprocess_sentence(x))
df.head()

[nltk_data] Error loading stopwords: <urlopen error [Errno -3]
[nltk_data]     Temporary failure in name resolution>
[nltk_data] Error loading punkt: <urlopen error [Errno -3] Temporary
[nltk_data]     failure in name resolution>


,review
0,one reviewers mentioned watching episode hooke...
1,wonderful little production . filming techniqu...
2,thought wonderful way spend time hot summer we...
3,basically family little boy jake thinks zombie...
4,petter mattei love time money visually stunnin...


In [5]:
import tensorflow as tf
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=10000)
tokenizer.fit_on_texts(df.review)

In [6]:
word2idx = tokenizer.word_index
idx2word = tokenizer.index_word
vocab_size = tokenizer.num_words

In [7]:
word2idx["great"]

17

In [8]:
idx2word[98000]

'schooly'

In [9]:
print(vocab_size)

10000


In [10]:
#print(tokenizer.word_index)

In [11]:
import numpy as np


def sentenceToData(tokens, WINDOW_SIZE):
    window = np.concatenate((np.arange(-WINDOW_SIZE,0),np.arange(1,WINDOW_SIZE+1)))
    X,Y=([],[])
    for word_index, word in enumerate(tokens) :
        if ((word_index - WINDOW_SIZE >= 0) and (word_index + WINDOW_SIZE <= len(tokens) - 1)) :
            X.append(word)
            Y.append([tokens[word_index-i] for i in window])
    return X, Y


WINDOW_SIZE = 5

X, Y = ([], [])
for review in df.review:
    for sentence in review.split("."):
        word_list = tokenizer.texts_to_sequences([sentence])[0]
        if len(word_list) >= WINDOW_SIZE:
            Y1, X1 = sentenceToData(word_list, WINDOW_SIZE//2)
            X.extend(X1)
            Y.extend(Y1)
    
X = np.array(X).astype(int)
y = np.array(Y).astype(int).reshape([-1,1])

In [12]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D

embedding_dim = 300
model = Sequential()
model.add(Embedding(vocab_size, embedding_dim))
model.add(GlobalAveragePooling1D())
model.add(Dense(vocab_size, activation='softmax'))

In [13]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X, y, batch_size = 128, epochs=5)

I0000 00:00:1787329411.661645      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/5
   53/23909 ━━━━━━━━━━━━━━━━━━━━ 1:10 3ms/step - accuracy: 0.0045 - loss: 9.2036

I0000 00:00:1787329414.559153      69 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


23909/23909 ━━━━━━━━━━━━━━━━━━━━ 74s 3ms/step - accuracy: 0.0410 - loss: 7.4063
Epoch 2/5
23909/23909 ━━━━━━━━━━━━━━━━━━━━ 71s 3ms/step - accuracy: 0.0662 - loss: 6.7522
Epoch 3/5
23909/23909 ━━━━━━━━━━━━━━━━━━━━ 72s 3ms/step - accuracy: 0.0775 - loss: 6.4480
Epoch 4/5
23909/23909 ━━━━━━━━━━━━━━━━━━━━ 72s 3ms/step - accuracy: 0.0849 - loss: 6.2468
Epoch 5/5
23909/23909 ━━━━━━━━━━━━━━━━━━━━ 71s 3ms/step - accuracy: 0.0906 - loss: 6.1042


In [14]:
history = model.fit(X, y, batch_size=128, epochs=10)

Epoch 1/10
23909/23909 ━━━━━━━━━━━━━━━━━━━━ 71s 3ms/step - accuracy: 0.0947 - loss: 6.0014
Epoch 2/10
23909/23909 ━━━━━━━━━━━━━━━━━━━━ 72s 3ms/step - accuracy: 0.0982 - loss: 5.9257
Epoch 3/10
23909/23909 ━━━━━━━━━━━━━━━━━━━━ 72s 3ms/step - accuracy: 0.1007 - loss: 5.8687
Epoch 4/10
23909/23909 ━━━━━━━━━━━━━━━━━━━━ 66s 3ms/step - accuracy: 0.1028 - loss: 5.8247
Epoch 5/10
23909/23909 ━━━━━━━━━━━━━━━━━━━━ 66s 3ms/step - accuracy: 0.1045 - loss: 5.7893
Epoch 6/10
23909/23909 ━━━━━━━━━━━━━━━━━━━━ 65s 3ms/step - accuracy: 0.1060 - loss: 5.7610
Epoch 7/10
23909/23909 ━━━━━━━━━━━━━━━━━━━━ 64s 3ms/step - accuracy: 0.1070 - loss: 5.7375
Epoch 8/10
23909/23909 ━━━━━━━━━━━━━━━━━━━━ 64s 3ms/step - accuracy: 0.1081 - loss: 5.7181
Epoch 9/10
23909/23909 ━━━━━━━━━━━━━━━━━━━━ 64s 3ms/step - accuracy: 0.1089 - loss: 5.7018
Epoch 10/10
23909/23909 ━━━━━━━━━━━━━━━━━━━━ 64s 3ms/step - accuracy: 0.1098 - loss: 5.6878


In [15]:
import pickle

# 1. Zapisujemy cały model Keras
model.save("word2vec.h5")

# 2. Zapisujemy Tokenizer (Niezbędny do Streamlit!)
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print("Model i Tokenizer zostały pomyślnie zapisane!")

Model i Tokenizer zostały pomyślnie zapisane!
